# Exploring feedback polynomials

Aaron J. Heller

This companion to [Gold codes from scratch](Gold_Code_from_Scratch.ipynb) explores feedback polynomials, recovers linear recurrences with Berlekamp–Massey, and compares the recurrences of Gold-family members.

## Sequence-generation setup
The helpers below generate a preferred pair and a sample of its Gold family. We use register length $n=15$ and 128 family members, with the preferred pair in the first two rows.


In [ ]:
from itertools import count, chain
from math import gcd

import numpy as np
from scipy import signal as sig


def q_generator(n, q_fn=1):
    """Generate good values of q, the decimation factor, for a given n."""

    # allow short hand for q(k)
    if q_fn == 1:
        q_fn = lambda k: 2**k + 1
    elif q_fn == 2:
        q_fn = lambda k: 2 ** (2 * k) - 2**k + 1

    # try out q_fn to see if it is really a function
    q_fn(1)

    # property 1
    if n % 4 == 0:
        raise ValueError("n mod 4 is 0!")

    # length of m-sequence
    N = 2**n - 1

    # good GCD for property 3
    good_gcd = 1 if n % 2 == 1 else 2

    qs = set()
    for k in count():
        # checking property 3
        if gcd(n, k) != good_gcd:
            continue

        # property 2
        q = q_fn(k)

        # the decimation is cyclic, so if we've cycled around, we're done
        q_mod_N = q % N
        if q_mod_N % 2 == 1:  # property 3, q is odd
            if q_mod_N in qs:
                break
            else:
                yield q_mod_N
                qs.add(q_mod_N)


def generate_good_qs(n):
    return chain(q_generator(n, 1), q_generator(n, 2))


def decimate_sequence(s, q):
    """Return cyclic decimation of the sequence s by the factor q."""
    N = len(s)
    i = np.mod(np.arange(N) * q, N)
    s_ = s[i]
    return s_


def gold_code_set(
    n, set_size, q=None, bipolar=True, include_pair=True, initial_phase=None
):
    """Generate set_size sequences, including the preferred pair if requested.

    initial_phase is the first cyclic shift of s2 used in the XOR sequences.
    None preserves the default starting shift of n + 1. When include_pair is
    true, the unshifted pair occupies the first two rows of the returned set.
    """
    if q is None:
        q = next(generate_good_qs(n))

    # the default sequence for length n
    s1 = sig.max_len_seq(n)[0]
    # the second sequence, forming a preferred pair with s1
    s2 = decimate_sequence(s1, q)

    phase = n + 1 if initial_phase is None else initial_phase
    pair_size = 2 if include_pair else 0
    rows = [s1, s2] if include_pair else []
    rows.extend(
        np.logical_xor(s1, np.roll(s2, p))
        for p in range(phase, phase + set_size - pair_size)
    )
    gc_set = np.vstack(rows).astype(s1.dtype, copy=False)
    if bipolar:
        gc_set = gc_set * 2 - 1

    return gc_set


In [ ]:
n = 15
set_size = 128
gc_set = gold_code_set(n, set_size, bipolar=True)
print(gc_set.shape)


## Exploring feedback polynomials


In [7]:
# this is somewhat faster than the implementation in sympy
#   http://fredrikj.net/python-flint/nmod_poly.html
#
# install:
# !conda install -c conda-forge python-flint
# or
# !pip install python-flint

    
try:
    import flint
except ImportError as ie:
    flint = False
    print('The package "flint" is not available. Skipping sections that need it.')

# from Timer import Timer

def taps_flint(n, algorithm='Cantor-Zassenhaus'):
    N = 2**n - 1
    # construct the poly x**N + 1 (mod 2)
    poly = flint.nmod_poly([1] + [0]*(N-1) + [1], mod=2)
    print(f"{poly}  GF{poly.modulus()}")
    factors = poly.factor(algorithm=algorithm)[1]
    fn = (fac for fac, mult in factors if fac.degree() == n)
    
    taps = [[ex for ex, c in enumerate(fac) if c][-2:1:-1] for fac in fn]
    
    return taps

The package "flint" is not available. Skipping sections that need it.


In [8]:
if flint:
    with Timer("Cantor-Zassenhaus"):
        fac = taps_flint(15, algorithm="Cantor-Zassenhaus")
        print(len(fac))

In [9]:
if flint:
    with Timer('Kaltofen-Shoup'):
        fac2 = taps_flint(15, algorithm='Kaltofen-Shoup')
        print(len(fac2))

### Recovering a recurrence with Berlekamp–Massey
The two preferred-pair members are m-sequences of linear complexity $n$. Their Gold XOR combinations have linear complexity $2n$: their minimal polynomial is the product of the two distinct degree-$n$ primitive polynomials. They are not m-sequences of order $2n$, since their period remains $2^n-1$.

Berlekamp–Massey recovers a minimal recurrence from at least twice its linear complexity in consecutive noiseless samples. The following experiments use full periods.


In [10]:
from importlib import reload
import lfsr
reload(lfsr)

# their first two members of the set are the preferred pair of m-sequences
[lfsr.run_berlekamp_massey(s) for s in gc_set[:2]==1]

ModuleNotFoundError: No module named 'lfsr'

### A shared recurrence for the XOR members
The XOR members share the product polynomial, so they can be generated by the same $2n$-stage linear recurrence with different initial states. The next cells compare their recovered coefficients, excluding the two original m-sequences, whose minimal polynomials have degree $n$.


In [ ]:
from joblib import delayed, Parallel, cpu_count
gc_taps = np.asarray(Parallel(n_jobs=cpu_count()//2, verbose=5)
                     (delayed(lfsr.run_berlekamp_massey)(s==1) 
                      for s in gc_set[2:]))  # exclude generator pair
gc_taps.shape

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:    2.6s
[Parallel(n_jobs=6)]: Done  60 tasks      | elapsed:   23.3s
[Parallel(n_jobs=6)]: Done 126 out of 126 | elapsed:   51.6s finished


(126, 30)

In [ ]:
# are the entries in each column of the array equal... yes!
np.all(np.diff(gc_taps, axis=0)==0)

True

### Feedback taps of decimated m-sequences


In [ ]:
import lfsr
import numpy as np

n=13
# the default sequence for length n
s1 = sig.max_len_seq(n)[0]
print(1, lfsr.run_berlekamp_massey(s1==1))
    
for i, q in enumerate(sorted(generate_good_qs(n))):
    s2 = decimate_sequence(s1, q)
    taps = lfsr.run_berlekamp_massey(s2==1)
    hex = np.packbits(np.array(taps, dtype=np.uint8))
    
    print(i, q, taps, hex)

1 [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1]
0 3 [1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0] [137 224]
1 3 [1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0] [137 224]
2 5 [1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0] [143 224]
3 5 [1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0] [143 224]
4 7 [1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1] [159 120]
5 9 [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1] [144  40]
6 13 [1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1] [215 136]
7 15 [1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1] [165  40]
8 17 [1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1] [135  24]
9 31 [1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1] [142  24]
10 33 [1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1] [209 232]
11 33 [1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1] [209 232]
12 41 [1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1] [184  72]
13 57 [1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0] [175  96]
14 59 [1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1] [133 168]
15 65 [1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0] [251 128]
16 67 [1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0] [202  96]
17 83 [1, 1, 0, 0, 0, 

In [ ]:
b0, b1=sig.max_len_seq(5, taps=[3])[0], sig.max_len_seq(5)[0]

lfsr.run_berlekamp_massey(b1)

b2 = sig.max_len_seq(5, taps=[4, 3, 2])[0]
b2
bm_taps2=np.asarray(lfsr.run_berlekamp_massey(b2))
print(mb_taps2)
print(np.arange(5)[bm_taps2==1])

b3 = sig.max_len_seq(5, taps=[4, 3, 1])[0]
b3
bm_taps3=np.asarray(lfsr.run_berlekamp_massey(b3))
bm_taps3
print(np.arange(5)[bm_taps3==1])

[1, 0, 1, 1, 1]
[0 2 3 4]
[0 1 3 4]
